# Phase 3 — CGRS: Curvature-Guided Rank Selection
## USC MS Computer Science (AI) — Spring 2026

**Algorithm:** Global rank adjustment using live diagonal Fisher λ_max vs tau threshold.

**tau values are taken directly from Phase 2 measured lambda_max:**
- `tau_A = 0.004680` → Phase 2 lambda_max at r=30 → targets flat zone (~90% acc)
- `tau_B = 0.013566` → Phase 2 lambda_max at r=12 → targets efficiency sweet spot (~89.3%)
- `tau_C = 0.051826` → Phase 2 lambda_max at r=10 → targets low-rank zone (~87-89%)

**Comparison:** Fixed r=6, Fixed r=10, Fixed r=16 (Phase 1 results, no retraining) vs CGRS × 3 taus.

**Estimated runtime:** A100 ≈ 8-10 min total (3 CGRS runs × ~3 min each). V100 ≈ 14-16 min.


In [3]:
# ── CELL 0: Fix NCCL conflict on USC CARC V100 nodes ──────────────────────
import os, sys

# Force PyTorch to use its own bundled NCCL, ignoring the broken system one
conda_prefix = os.environ.get("CONDA_PREFIX", "")
torch_lib    = f"{conda_prefix}/lib/python3.10/site-packages/torch/lib"
conda_lib    = f"{conda_prefix}/lib"

os.environ["LD_PRELOAD"]      = ""
os.environ["LD_LIBRARY_PATH"] = f"{torch_lib}:{conda_lib}:" + os.environ.get("LD_LIBRARY_PATH", "")

print(f"CONDA_PREFIX   : {conda_prefix}")
print(f"LD_PRELOAD     : (cleared)")
print(f"LD_LIBRARY_PATH: {os.environ['LD_LIBRARY_PATH'][:100]}...")

CONDA_PREFIX   : 
LD_PRELOAD     : (cleared)
LD_LIBRARY_PATH: /lib/python3.10/site-packages/torch/lib:/lib:...


In [2]:
pip install torch torchvision transformers peft scipy numpy --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [1]:
# ----------------------------------------------------------------
# CELL 2 — Imports and reproducibility
# Identical seed setup to Phases 1 and 2 for consistent splits.
# ----------------------------------------------------------------
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from transformers import ViTForImageClassification, ViTImageProcessor
from peft import LoraConfig, get_peft_model
from scipy.stats import pearsonr, spearmanr
from collections import defaultdict

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cpu


/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
# ----------------------------------------------------------------
# CELL 3 — Config + load Phase 1/2 results + tau calibration
# ----------------------------------------------------------------
PHASE1_DIR = "./phase1_results"   # config.json, results.json, lora_r{r}_adapter/, lora_r{r}_full.pt
PHASE2_DIR = "./phase2_results"   # all_results_complete.json
PHASE3_DIR = "./phase3_results"   # all outputs go here
os.makedirs(PHASE3_DIR, exist_ok=True)

# Load Phase 1 config and results
with open(f"{PHASE1_DIR}/config.json") as f:
    CONFIG = json.load(f)
with open(f"{PHASE1_DIR}/results.json") as f:
    phase1_results = json.load(f)

LORA_RANKS = [3, 5, 6, 10, 12, 16, 30, 52, 64, 80, 100, 128, 150, 200]
CONFIG["lora_ranks"] = LORA_RANKS

# ---- Load Phase 2 curvature results (for tau calibration) ----
p2_complete_path = f"{PHASE2_DIR}/all_results_complete.json"
phase2_available = os.path.exists(p2_complete_path)

if phase2_available:
    with open(p2_complete_path) as f:
        all_results_p2 = json.load(f)
    curvature_data = all_results_p2.get("curvature", {})
    print("Phase 2 results loaded — computing tau from actual lambda_max values.")
    lmax_by_rank = {}
    for r in LORA_RANKS:
        key = f"r={r}"
        if key in curvature_data:
            lmax_by_rank[r] = curvature_data[key]["lambda_max"]
    if lmax_by_rank:
        lmax_values = list(lmax_by_rank.values())
        print(f"  lambda_max range: {min(lmax_values):.6f} — {max(lmax_values):.6f}")
        sorted_lmax = sorted(lmax_values)
        n = len(sorted_lmax)
        tau_high = sorted_lmax[n // 4]
        tau_mid  = sorted_lmax[n // 2]
        tau_low  = sorted_lmax[3 * n // 4]
        TAU_VALUES = [tau_high, tau_mid, tau_low]
        print(f"  Calibrated tau: high={tau_high:.6f}, mid={tau_mid:.6f}, low={tau_low:.6f}")
    else:
        print("  Warning: curvature_data empty — using default tau values.")
        TAU_VALUES = [0.01, 0.05, 0.10]
        lmax_by_rank = {}
else:
    print("Phase 2 results not found — using default tau values.")
    print("  Place phase2_results/all_results_complete.json for calibrated tau.")
    TAU_VALUES = [0.01, 0.05, 0.10]
    lmax_by_rank = {}
    curvature_data = {}

# ---- CGRS hyperparameters ----
CONFIG["cgrs_check_every"]   = 200
CONFIG["cgrs_cooldown"]      = 100
CONFIG["cgrs_probe_batches"] = 8
CONFIG["cgrs_r_init"]        = 16
CONFIG["cgrs_r_min"]         = 3
CONFIG["cgrs_r_max"]         = 64
CONFIG["cgrs_rank_list"]     = [r for r in LORA_RANKS if r <= CONFIG["cgrs_r_max"]]
CONFIG["cgrs_tau_values"]    = TAU_VALUES

with open(f"{PHASE3_DIR}/config_phase3.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"\nCGRS config:")
print(f"  r_init       : {CONFIG['cgrs_r_init']}")
print(f"  r_min / r_max: {CONFIG['cgrs_r_min']} / {CONFIG['cgrs_r_max']}")
print(f"  rank_list    : {CONFIG['cgrs_rank_list']}")
print(f"  check_every  : {CONFIG['cgrs_check_every']} steps")
print(f"  cooldown     : {CONFIG['cgrs_cooldown']} steps")
print(f"  probe_batches: {CONFIG['cgrs_probe_batches']} (={CONFIG['cgrs_probe_batches']*4} samples)")
print(f"  tau_values   : {[f'{t:.4f}' for t in TAU_VALUES]}")
print(f"\nPhase 1 baseline results:")
for k, v in phase1_results.items():
    p = v.get("trainable_params", "n/a")
    print(f"  {k:<22} -> Test Acc: {v['test_acc']:.2f}%  Params: {p}")

Phase 2 results loaded — computing tau from actual lambda_max values.
  lambda_max range: 0.000038 — 0.226160
  Calibrated tau: high=0.000205, mid=0.004680, low=0.051826

CGRS config:
  r_init       : 16
  r_min / r_max: 3 / 64
  rank_list    : [3, 5, 6, 10, 12, 16, 30, 52, 64]
  check_every  : 200 steps
  cooldown     : 100 steps
  probe_batches: 8 (=32 samples)
  tau_values   : ['0.0002', '0.0047', '0.0518']

Phase 1 baseline results:
  Full fine-tune         -> Test Acc: 92.87%  Params: 86567396
  Frozen backbone        -> Test Acc: 86.46%  Params: 76900
  LoRA r=3               -> Test Acc: 84.77%  Params: 110592
  LoRA r=5               -> Test Acc: 87.36%  Params: 184320
  LoRA r=6               -> Test Acc: 87.27%  Params: 221184
  LoRA r=10              -> Test Acc: 89.02%  Params: 368640
  LoRA r=12              -> Test Acc: 89.31%  Params: 442368
  LoRA r=16              -> Test Acc: 89.51%  Params: 589824
  LoRA r=30              -> Test Acc: 90.30%  Params: 1105920
  LoRA r

In [3]:
# ----------------------------------------------------------------
# CELL 4 — Dataset and DataLoaders
# ViT-Base/16 always uses mean=[0.5,0.5,0.5] std=[0.5,0.5,0.5].
# The lora_r{r}_adapter folders only store LoRA weights — no
# preprocessor_config.json. Try HuggingFace first; fallback to
# hardcoded values (identical to what Phase 1 & 2 used).
# ----------------------------------------------------------------

try:
    processor = ViTImageProcessor.from_pretrained(CONFIG["model_name"])
    mean = processor.image_mean
    std  = processor.image_std
    print(f"Processor loaded from HuggingFace: {CONFIG['model_name']}")
except Exception:
    mean = [0.5, 0.5, 0.5]
    std  = [0.5, 0.5, 0.5]
    print("No HuggingFace access — using hardcoded ViT-Base normalization.")

print(f"  Normalization mean: {mean}")
print(f"  Normalization std : {std}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

print("Loading CIFAR-100...")
full_train   = datasets.CIFAR100(root="./data", train=True,  download=True, transform=transform)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

val_size = len(full_train) - CONFIG["train_size"]
train_dataset, val_dataset = random_split(
    full_train,
    [CONFIG["train_size"], val_size],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"], shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=CONFIG["batch_size"], shuffle=False,
                          num_workers=2, pin_memory=True)
curvature_probe_loader = DataLoader(train_dataset, batch_size=4, shuffle=False,
                                    num_workers=2, pin_memory=True)

print(f"  Train : {len(train_dataset):,}")
print(f"  Val   : {len(val_dataset):,}")
print(f"  Test  : {len(test_dataset):,}")
print(f"  Probe loader: batch_size=4 | {len(curvature_probe_loader)} batches total")

Processor loaded from HuggingFace: google/vit-base-patch16-224
  Normalization mean: (0.5, 0.5, 0.5)
  Normalization std : (0.5, 0.5, 0.5)
Loading CIFAR-100...
  Train : 45,000
  Val   : 5,000
  Test  : 10,000
  Probe loader: batch_size=4 | 11250 batches total


In [4]:
# ----------------------------------------------------------------
# CELL 5 — Model builders and evaluation
# ----------------------------------------------------------------
def count_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total


def build_lora_model_silent(r):
    """Build LoRA ViT-Base with rank r. No console output."""
    base = ViTForImageClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=CONFIG["num_classes"],
        ignore_mismatched_sizes=True,
    )
    cfg = LoraConfig(
        r             = r,
        lora_alpha    = CONFIG["lora_alpha"],
        lora_dropout  = CONFIG["lora_dropout"],
        target_modules= CONFIG["target_modules"],
        bias          = "none",
    )
    return get_peft_model(base, cfg).to(device)


def build_lora_model_verbose(r):
    """Build LoRA ViT-Base with rank r. Prints trainable params."""
    model = build_lora_model_silent(r)
    t, total = count_parameters(model)
    print(f"  LoRA r={r:<4} | Trainable: {t:>10,} / {total:,} ({100*t/total:.3f}%)")
    return model, t


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    loss_sum, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(images)
        loss_sum += criterion(outputs.logits, labels).item()
        correct  += (outputs.logits.argmax(dim=1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / len(loader), 100.0 * correct / total


print("Model builders and evaluate() ready.")

Model builders and evaluate() ready.


In [5]:
# ----------------------------------------------------------------
# CELL 6 — Rank transition and Fisher probe utilities
# ----------------------------------------------------------------
_criterion_probe = nn.CrossEntropyLoss()


def transition_lora_rank(old_model, old_r, new_r):
    """
    Build a new LoRA model at new_r, initialized from old_model (old_r).
    Increase → zero-pad A and B matrices.
    Decrease → SVD truncation of B@A, keep top new_r components.
    Returns the new model on device.
    """
    old_params = {name: param.data.clone().cpu()
                  for name, param in old_model.named_parameters()}

    new_model = build_lora_model_silent(new_r)
    new_model.cpu()

    # Step 1: Copy all non-LoRA parameters
    new_param_dict = dict(new_model.named_parameters())
    for name, old_data in old_params.items():
        if 'lora_A' not in name and 'lora_B' not in name:
            if name in new_param_dict:
                new_param_dict[name].data.copy_(old_data)

    # Step 2: Transfer LoRA matrices with rank adjustment
    lora_bases = {}
    for name, data in old_params.items():
        if 'lora_A' in name:
            base = name[:name.index('lora_A')]
            lora_bases.setdefault(base, {})
            lora_bases[base]['A']      = data
            lora_bases[base]['name_A'] = name
        elif 'lora_B' in name:
            base = name[:name.index('lora_B')]
            lora_bases.setdefault(base, {})
            lora_bases[base]['B']      = data
            lora_bases[base]['name_B'] = name

    for base, pair in lora_bases.items():
        if 'A' not in pair or 'B' not in pair:
            continue

        A = pair['A'].float()   # (old_r, in_features)
        B = pair['B'].float()   # (out_features, old_r)

        if new_r > old_r:
            # Zero-padding
            new_A = torch.zeros(new_r, A.shape[1])
            new_A[:old_r, :] = A
            new_B = torch.zeros(B.shape[0], new_r)
            new_B[:, :old_r] = B
        else:
            # SVD truncation
            W = B @ A
            try:
                U, S, Vh = torch.linalg.svd(W, full_matrices=False)
                sqrt_S = torch.sqrt(S[:new_r].clamp(min=0.0))
                new_B  = U[:, :new_r] * sqrt_S
                new_A  = Vh[:new_r, :] * sqrt_S.unsqueeze(1)
            except RuntimeError:
                new_A = A[:new_r, :]
                new_B = B[:, :new_r]

        for name, param in new_model.named_parameters():
            if name.startswith(base) and 'lora_A' in name:
                param.data.copy_(new_A.to(param.dtype))
            elif name.startswith(base) and 'lora_B' in name:
                param.data.copy_(new_B.to(param.dtype))

    return new_model.to(device)


def probe_fisher(model, probe_loader, n_batches):
    """
    Cheap diagonal Fisher probe for CGRS rank-check step.
    Returns lambda_max (scalar float).
    """
    was_training = model.training
    model.eval()
    fisher_diag  = None
    batches_done = 0

    for images, labels in probe_loader:
        if batches_done >= n_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        model.zero_grad()
        outputs = model(images)
        loss    = _criterion_probe(outputs.logits, labels)
        loss.backward()
        with torch.no_grad():
            grads = torch.cat([
                p.grad.detach().pow(2).flatten()
                for p in model.parameters()
                if p.requires_grad and p.grad is not None
            ])
        fisher_diag = grads if fisher_diag is None else fisher_diag + grads
        batches_done += 1

    model.zero_grad()
    if was_training:
        model.train()

    if fisher_diag is None or batches_done == 0:
        return float('inf')

    fisher_diag /= batches_done
    return float(fisher_diag.max().item())


print("transition_lora_rank() and probe_fisher() ready.")
print("  transition_lora_rank: zero-pad (increase) / SVD truncation (decrease)")
print(f"  probe_fisher: {CONFIG['cgrs_probe_batches']} batches x 4 = {CONFIG['cgrs_probe_batches']*4} samples per check")

transition_lora_rank() and probe_fisher() ready.
  transition_lora_rank: zero-pad (increase) / SVD truncation (decrease)
  probe_fisher: 8 batches x 4 = 32 samples per check


In [6]:
# ----------------------------------------------------------------
# CELL 7 — CGRS training loop (fully resume-safe)
# ----------------------------------------------------------------
CKPT_BASE = f"{PHASE3_DIR}/checkpoints"
os.makedirs(CKPT_BASE, exist_ok=True)


def _ckpt_dir(label):
    return f"{CKPT_BASE}/{label}"


def _save(label, payload, model, optimizer, scheduler):
    """Save full training state after every rank change or epoch."""
    d = _ckpt_dir(label)
    os.makedirs(d, exist_ok=True)
    torch.save(model.state_dict(),     f"{d}/model.pt")
    torch.save(optimizer.state_dict(), f"{d}/optimizer.pt")
    torch.save(scheduler.state_dict(), f"{d}/scheduler.pt")
    payload_json = {**payload,
                    "rank_step_count": {str(k): v
                                        for k, v in payload["rank_step_count"].items()}}
    with open(f"{d}/state.json", "w") as f:
        json.dump(payload_json, f, indent=2)


def _load(label):
    """Load saved state. Returns (state_dict, ckpt_dir) or None."""
    d = _ckpt_dir(label)
    if not os.path.exists(f"{d}/state.json"):
        return None
    with open(f"{d}/state.json") as f:
        state = json.load(f)
    state["rank_step_count"] = defaultdict(
        int, {int(k): v for k, v in state["rank_step_count"].items()}
    )
    return state, d


def run_cgrs(tau, label):
    RANK_LIST = CONFIG["cgrs_rank_list"]
    R_MIN     = CONFIG["cgrs_r_min"]
    R_MAX     = CONFIG["cgrs_r_max"]
    K         = CONFIG["cgrs_check_every"]
    COOLDOWN  = CONFIG["cgrs_cooldown"]
    N_PROBE   = CONFIG["cgrs_probe_batches"]
    r_init    = CONFIG["cgrs_r_init"]
    EPOCHS    = CONFIG["epochs"]
    criterion = nn.CrossEntropyLoss()
    total_steps = EPOCHS * len(train_loader)

    print(f"\n{'='*65}")
    print(f"  {label} | tau={tau:.4f} | r_init={r_init}")
    print(f"  check_every={K} | cooldown={COOLDOWN} | probe={N_PROBE} batches")
    print(f"{'='*65}")

    # ---- RESUME or FRESH START ----
    resume = _load(label)

    if resume is not None:
        state, ckpt_dir    = resume
        start_epoch        = state["start_epoch"]
        start_batch        = state["start_batch"]
        current_r          = state["current_r"]
        global_step        = state["global_step"]
        steps_since_chg    = state["steps_since_chg"]
        rank_trajectory    = state["rank_trajectory"]
        lambda_trajectory  = state["lambda_trajectory"]
        rank_changes       = state["rank_changes"]
        rank_step_count    = state["rank_step_count"]
        resume_loss        = state.get("running_loss", 0.0)
        resume_correct     = state.get("correct", 0)
        resume_total       = state.get("total", 0)

        print(f"  [RESUME] epoch={start_epoch} batch={start_batch} r={current_r} step={global_step}")

        model, _ = build_lora_model_verbose(current_r)
        model.load_state_dict(torch.load(f"{ckpt_dir}/model.pt", map_location=device))
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"]
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
        optimizer.load_state_dict(torch.load(f"{ckpt_dir}/optimizer.pt", map_location=device))
        scheduler.load_state_dict(torch.load(f"{ckpt_dir}/scheduler.pt"))
    else:
        start_epoch       = 1
        start_batch       = 0
        current_r         = r_init
        global_step       = 0
        steps_since_chg   = COOLDOWN
        rank_trajectory   = []
        lambda_trajectory = []
        rank_changes      = []
        rank_step_count   = defaultdict(int)
        resume_loss       = 0.0
        resume_correct    = 0
        resume_total      = 0

        print(f"  [FRESH] Starting from epoch=1, r={r_init}")
        torch.cuda.empty_cache()
        model, _ = build_lora_model_verbose(r_init)
        optimizer = optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CONFIG["lr_lora"], weight_decay=CONFIG["weight_decay"]
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    # ---- TRAINING LOOP ----
    for epoch in range(start_epoch, EPOCHS + 1):
        model.train()

        if epoch == start_epoch and start_batch > 0:
            running_loss = resume_loss
            correct      = resume_correct
            total        = resume_total
        else:
            running_loss, correct, total = 0.0, 0, 0

        for batch_idx, (images, labels_b) in enumerate(train_loader):

            if epoch == start_epoch and batch_idx < start_batch:
                continue

            images   = images.to(device, non_blocking=True)
            labels_b = labels_b.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs.logits, labels_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0
            )
            optimizer.step()
            scheduler.step()

            running_loss    += loss.item()
            correct         += (outputs.logits.argmax(1) == labels_b).sum().item()
            total           += labels_b.size(0)
            rank_step_count[current_r] += 1
            global_step     += 1
            steps_since_chg += 1

            # ---- RANK CHECK ----
            if steps_since_chg >= K:
                lmax = probe_fisher(model, curvature_probe_loader, N_PROBE)
                rank_trajectory.append((global_step, current_r))
                lambda_trajectory.append((global_step, lmax))

                rank_idx = (RANK_LIST.index(current_r) if current_r in RANK_LIST
                            else min(range(len(RANK_LIST)),
                                     key=lambda i: abs(RANK_LIST[i] - current_r)))
                new_r  = current_r
                reason = "stable"

                if lmax > tau and rank_idx < len(RANK_LIST) - 1:
                    candidate = RANK_LIST[rank_idx + 1]
                    if candidate <= R_MAX:
                        new_r  = candidate
                        reason = f"sharp (lmax={lmax:.5f} > tau={tau:.5f})"

                elif lmax < tau and rank_idx > 0:
                    candidate = RANK_LIST[rank_idx - 1]
                    if candidate >= R_MIN:
                        new_r  = candidate
                        reason = f"flat (lmax={lmax:.5f} < tau={tau:.5f})"

                if new_r != current_r:
                    direction = "UP" if new_r > current_r else "DOWN"
                    print(f"  [Step {global_step:>5}] Rank {direction}: {current_r} → {new_r} | {reason}")

                    rank_changes.append({
                        "step"  : global_step,
                        "old_r" : current_r,
                        "new_r" : new_r,
                        "lambda": lmax,
                        "reason": reason,
                    })

                    current_lr = float(scheduler.get_last_lr()[0])
                    model      = transition_lora_rank(model, current_r, new_r)
                    current_r  = new_r
                    remaining  = max(total_steps - global_step, 1)
                    optimizer  = optim.AdamW(
                        filter(lambda p: p.requires_grad, model.parameters()),
                        lr=current_lr, weight_decay=CONFIG["weight_decay"]
                    )
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining)
                    torch.cuda.empty_cache()

                    _save(label, {
                        "start_epoch"    : epoch,
                        "start_batch"    : batch_idx + 1,
                        "current_r"      : current_r,
                        "global_step"    : global_step,
                        "steps_since_chg": 0,
                        "rank_trajectory": rank_trajectory,
                        "lambda_trajectory": lambda_trajectory,
                        "rank_changes"   : rank_changes,
                        "rank_step_count": rank_step_count,
                        "running_loss"   : running_loss,
                        "correct"        : correct,
                        "total"          : total,
                    }, model, optimizer, scheduler)
                    print(f"  [SAVED] After rank change at step {global_step}")

                steps_since_chg = 0

        # ---- END OF EPOCH ----
        val_loss, val_acc = evaluate(model, val_loader)
        train_acc = 100.0 * correct / total
        print(f"\n  {label} Epoch {epoch}/{EPOCHS} r={current_r} | "
              f"Train Loss: {running_loss/len(train_loader):.4f} Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.2f}%")

        _save(label, {
            "start_epoch"    : epoch + 1,
            "start_batch"    : 0,
            "current_r"      : current_r,
            "global_step"    : global_step,
            "steps_since_chg": steps_since_chg,
            "rank_trajectory": rank_trajectory,
            "lambda_trajectory": lambda_trajectory,
            "rank_changes"   : rank_changes,
            "rank_step_count": rank_step_count,
            "running_loss"   : 0.0,
            "correct"        : 0,
            "total"          : 0,
        }, model, optimizer, scheduler)
        print(f"  [SAVED] Epoch {epoch} checkpoint.")
        start_batch = 0

    # ---- FINAL EVALUATION ----
    test_loss, test_acc = evaluate(model, test_loader)
    trainable_params    = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_steps_done    = sum(rank_step_count.values())
    avg_rank            = sum(r * cnt for r, cnt in rank_step_count.items()) / max(total_steps_done, 1)

    print(f"\n  {label} DONE — Test Acc: {test_acc:.2f}% | Avg rank: {avg_rank:.2f} | Changes: {len(rank_changes)}")

    del model
    torch.cuda.empty_cache()

    return {
        "test_acc"        : test_acc,
        "test_loss"       : test_loss,
        "tau"             : tau,
        "r_init"          : r_init,
        "final_rank"      : current_r,
        "avg_rank"        : avg_rank,
        "trainable_params": trainable_params,
        "n_rank_changes"  : len(rank_changes),
        "rank_changes"    : rank_changes,
        "rank_step_count" : dict(rank_step_count),
        "rank_trajectory" : rank_trajectory,
        "lambda_trajectory": lambda_trajectory,
    }


print("run_cgrs() ready — saves after every rank change and every epoch.")

run_cgrs() ready — saves after every rank change and every epoch.


In [7]:
# ----------------------------------------------------------------
# CELL 8 — Load Phase 1 fixed-rank baselines
# ----------------------------------------------------------------
phase3_results = {}

BASELINE_RANKS = [6, 10, 16]

print("=" * 70)
print("  FIXED-RANK BASELINES from Phase 1")
print("=" * 70)

for r in BASELINE_RANKS:
    key_p1 = f"LoRA r={r}"
    if key_p1 in phase1_results:
        v = phase1_results[key_p1]
        phase3_results[f"Fixed r={r}"] = {
            "test_acc"        : v["test_acc"],
            "test_loss"       : v["test_loss"],
            "trainable_params": v.get("trainable_params", "n/a"),
            "source"          : "Phase 1",
            "avg_rank"        : float(r),
        }
        print(f"  Fixed r={r:<5} - Test Acc: {v['test_acc']:.2f}%  Test Loss: {v['test_loss']:.4f}")
    else:
        print(f"  Fixed r={r} NOT FOUND in phase1_results — check results.json")

for key in ["Full fine-tune", "Frozen backbone"]:
    if key in phase1_results:
        v = phase1_results[key]
        phase3_results[key] = {
            "test_acc" : v["test_acc"],
            "test_loss": v["test_loss"],
            "source"   : "Phase 1",
        }
        print(f"  {key:<22} - Test Acc: {v['test_acc']:.2f}%")

with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
    json.dump(phase3_results, f, indent=2)

print(f"\nBaselines loaded. phase3_results.json saved to {PHASE3_DIR}/")

  FIXED-RANK BASELINES from Phase 1
  Fixed r=6     - Test Acc: 87.27%  Test Loss: 0.6846
  Fixed r=10    - Test Acc: 89.02%  Test Loss: 0.5439
  Fixed r=16    - Test Acc: 89.51%  Test Loss: 0.4606
  Full fine-tune         - Test Acc: 92.87%
  Frozen backbone        - Test Acc: 86.46%

Baselines loaded. phase3_results.json saved to ./phase3_results/


In [ ]:
# ----------------------------------------------------------------
# CELL 9 — CGRS Run 1: tau_high
# ----------------------------------------------------------------
tau1   = TAU_VALUES[0]
label1 = f"CGRS_tau{tau1:.4f}"

if label1 in phase3_results and "test_acc" in phase3_results[label1]:
    print(f"Already complete: {label1}  acc={phase3_results[label1]['test_acc']:.2f}%")
else:
    result1 = run_cgrs(tau=tau1, label=label1)
    phase3_results[label1] = result1
    with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
        json.dump(phase3_results, f, indent=2)
    with open(f"{PHASE3_DIR}/trajectory_{label1}.json", "w") as f:
        json.dump({
            "rank_trajectory"  : result1["rank_trajectory"],
            "lambda_trajectory": result1["lambda_trajectory"],
            "rank_changes"     : result1["rank_changes"],
            "rank_step_count"  : result1["rank_step_count"],
        }, f, indent=2)
    print(f"Saved phase3_results.json")


  CGRS_tau0.0002 | tau=0.0002 | r_init=16
  check_every=200 | cooldown=100 | probe=8 batches
  [FRESH] Starting from epoch=1, r=16


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:06<00:00, 29.07it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  LoRA r=16   | Trainable:    589,824 / 86,465,380 (0.682%)


/home1/jkshaikh/.conda/envs/cgrs_env/lib/python3.10/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  [Step   100] Rank UP: 16 → 30 | sharp (lmax=0.00139 > tau=0.00020)


...

  [SAVED] After rank change at step 100
  [Step   300] Rank UP: 30 → 52 | sharp (lmax=0.00238 > tau=0.00020)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 3730.44it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED] After rank change at step 300
  [Step   500] Rank UP: 52 → 64 | sharp (lmax=0.00163 > tau=0.00020)


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 4802.02it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


  [SAVED] After rank change at step 500


In [ ]:
# ----------------------------------------------------------------
# CELL 10 — CGRS Run 2: tau_mid
# ----------------------------------------------------------------
tau2   = TAU_VALUES[1]
label2 = f"CGRS_tau{tau2:.4f}"

if label2 in phase3_results and "test_acc" in phase3_results[label2]:
    print(f"Already complete: {label2}  acc={phase3_results[label2]['test_acc']:.2f}%")
else:
    result2 = run_cgrs(tau=tau2, label=label2)
    phase3_results[label2] = result2
    with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
        json.dump(phase3_results, f, indent=2)
    with open(f"{PHASE3_DIR}/trajectory_{label2}.json", "w") as f:
        json.dump({
            "rank_trajectory"  : result2["rank_trajectory"],
            "lambda_trajectory": result2["lambda_trajectory"],
            "rank_changes"     : result2["rank_changes"],
            "rank_step_count"  : result2["rank_step_count"],
        }, f, indent=2)
    print(f"Saved phase3_results.json")

In [ ]:
# ----------------------------------------------------------------
# CELL 11 — CGRS Run 3: tau_low
# ----------------------------------------------------------------
tau3   = TAU_VALUES[2]
label3 = f"CGRS_tau{tau3:.4f}"

if label3 in phase3_results and "test_acc" in phase3_results[label3]:
    print(f"Already complete: {label3}  acc={phase3_results[label3]['test_acc']:.2f}%")
else:
    result3 = run_cgrs(tau=tau3, label=label3)
    phase3_results[label3] = result3
    with open(f"{PHASE3_DIR}/phase3_results.json", "w") as f:
        json.dump(phase3_results, f, indent=2)
    with open(f"{PHASE3_DIR}/trajectory_{label3}.json", "w") as f:
        json.dump({
            "rank_trajectory"  : result3["rank_trajectory"],
            "lambda_trajectory": result3["lambda_trajectory"],
            "rank_changes"     : result3["rank_changes"],
            "rank_step_count"  : result3["rank_step_count"],
        }, f, indent=2)
    print(f"Saved phase3_results.json")

In [ ]:
# ----------------------------------------------------------------
# CELL 12 — Status check: which CGRS runs are complete?
# ----------------------------------------------------------------
print("=== Checking existing results ===\n")

TAU_LABELS = [f"CGRS_tau{t:.4f}" for t in TAU_VALUES]

for label in TAU_LABELS:
    if label in phase3_results and "test_acc" in phase3_results[label]:
        acc   = phase3_results[label]["test_acc"]
        avg_r = phase3_results[label].get("avg_rank", "?")
        print(f"  {label:25s} -> COMPLETE ✓  | Test Acc: {acc:.2f}%  | Avg rank: {avg_r:.2f}")
    else:
        ckpt = f"{PHASE3_DIR}/checkpoints/{label}/state.json"
        if os.path.exists(ckpt):
            with open(ckpt) as f:
                s = json.load(f)
            status = f"IN PROGRESS (epoch={s.get('start_epoch','?')} step={s.get('global_step','?')} r={s.get('current_r','?')})"
        else:
            status = "NOT STARTED"
        print(f"  {label:25s} -> {status}")

print()
for label in TAU_LABELS:
    if label not in phase3_results or "test_acc" not in phase3_results[label]:
        print(f"  → Run the corresponding cell to train: {label}")

In [ ]:
# ── CELL 12: Correlation analysis — verify Phase 2 hypothesis on CGRS data ──
# Does the curvature-accuracy relationship (R²=0.9636 from Phase 2)
# also hold for the CGRS runs? We check lambda_max at end of each run vs acc.

from scipy.stats import pearsonr, spearmanr

# Collect final lambda_max (last recorded value) and test acc for each CGRS run
cgrs_lmax_vals = []
cgrs_acc_vals  = []
for label, res in cgrs_results.items():
    ltraj = res["lambda_trajectory"]
    if ltraj:
        last_lmax = ltraj[-1][1]
        cgrs_lmax_vals.append(last_lmax)
        cgrs_acc_vals.append(res["test_acc"])
        print(f"  {label}: final_lmax={last_lmax:.6f}  acc={res['test_acc']:.2f}%")

if len(cgrs_lmax_vals) >= 3:
    r_p, p_p = pearsonr(cgrs_lmax_vals, cgrs_acc_vals)
    r_s, p_s = spearmanr(cgrs_lmax_vals, cgrs_acc_vals)
    print(f"\nCGRS Pearson  r={r_p:.4f}  p={p_p:.4f}")
    print(f"CGRS Spearman r={r_s:.4f}  p={p_s:.4f}")
    print(f"(Phase 2 had R²={0.9636:.4f} — compare with CGRS correlation)")
else:
    print("Not enough data points for correlation (need ≥ 3 CGRS runs).")


In [ ]:
# ── CELL 13 (OPTIONAL): Per-layer CGRS ─────────────────────────────────────
# If time permits, assign each transformer layer its own rank based on
# its local lambda_max from Phase 2 per-layer analysis.
# Deeper layers (9-11) showed higher curvature → higher rank.
# Shallow layers (0-3) showed lower curvature → lower rank.
# This maximises parameter efficiency per layer.
#
# Skip this cell if time is limited — it is a bonus contribution.

print("Per-layer CGRS is a bonus Task 7 — implement only if time permits.")
print("See Phase 2 per-layer heatmaps for layer-wise lambda_max values.")
print("Approach:")
print("  1. Load per-layer lambda_max from Phase 2 curvature results.")
print("  2. Assign each of the 12 layers a starting rank proportional to its lambda_max.")
print("  3. Run a separate CGRS scheduler per layer during training.")
print("  4. Compare: global CGRS vs per-layer CGRS in the final accuracy/params table.")
